In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
from datasets import load_dataset

# Download full metamath subset
dataset = load_dataset("meta-math/MetaMathQA", split="train")

# Save to disk
dataset.save_to_disk("/home/guy.bilitski/peft/notebooks/GSM8")


Saving the dataset (1/1 shards): 100%|██████████| 395000/395000 [00:00<00:00, 1011485.84 examples/s]


In [3]:
dataset[0]

{'type': 'MATH_AnsAug',
 'query': "Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?",
 'original_question': "Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?",
 'response': "The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in the complex plane is given by the formula $\\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$.\nIn this case, Joe's point is $(1,2)$ and Gracie's point is $(-1,1)$.\nSo the distance between their points is $\\sqrt{((-1)-(1))^2+((1)-(2))^2}=\\sqrt{(-2)^2+(-1)^2}=\\sqrt{4+1}=\\sqrt{5}$.\nTherefore, Gracie and Joe's points are $\\boxed{\\sqrt{5}}$ units apart.\nThe answer is: \\sqrt{5}"}

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn

import transformers
from transformers import (AutoModelForCausalLM, 
                          AutoTokenizer, 
                          BitsAndBytesConfig, 
                          TrainingArguments, 
                          pipeline, 
                          logging)
from datasets import Dataset
from peft import LoraConfig, PeftConfig
import bitsandbytes as bnb
from trl import SFTTrainer

from sklearn.metrics import (accuracy_score, 
                             classification_report, 
                             confusion_matrix)
from sklearn.model_selection import train_test_split

2024-05-08 00:20:02.609083: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-08 00:20:02.609198: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-08 00:20:02.744446: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
# model_name = "/kaggle/input/gemma-finetuned-gsm8k/transformers/finetuned_on_10/1"
model_name = "/kaggle/input/gemma/transformers/7b-it/1"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config, 
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
model_name = "/kaggle/input/gemma/transformers/7b-it/1"

max_seq_length = 2048
tokenizer = AutoTokenizer.from_pretrained(model_name, max_seq_length=max_seq_length)
EOS_TOKEN = tokenizer.eos_token

In [9]:
filename = "/kaggle/input/grade-school-math-8k-q-a/main_train.csv"

df = pd.read_csv(filename)
df

,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...
...,...,...
7468,"Very early this morning, Elise left home in a ...","For the distance she traveled, Elise paid 23 -..."
7469,Josh is saving up for a box of cookies. To rai...,He makes $.5 profit on each bracelet because 1...
7470,Colin can skip at six times the speed that Bra...,Tony can skip at twice the speed that Bruce ca...
7471,"Janet, a third grade teacher, is picking up th...",Janet needs 35 lunches for the kids + 5 for th...


In [10]:
trainfilename = "/kaggle/input/grade-school-math-8k-q-a/main_train.csv"
testfilename = "/kaggle/input/grade-school-math-8k-q-a/main_test.csv"

traindf = pd.read_csv(trainfilename)
# traindf = traindf.drop(['title'],axis=1)

traindf, evaldf = train_test_split(df, test_size=0.2, random_state=42)

testdf = pd.read_csv(testfilename)
# testdf = testdf.drop(['title'],axis=1)

#selecting the firsst 100 rows only
# testdf = testdf[:50]
evaldf = evaldf[:9]
traindf = traindf[:9]

In [11]:
def few_shot_string_generator(traindf, number_of_shots):
    ans = ""
    for x in range(number_of_shots):
        random_row = traindf.sample(n=1).iloc[0]
        ans+=f"Question: {random_row['question']}\n"
        ans+=f"Solution: {random_row['answer']}\n\n"
    
    return ans

In [12]:
few_shot_string = ""
def generate_prompt_gsm8k(data_point):
    return f"""{data_point['question']} [SEP] {data_point['answer']}
            """.strip() + EOS_TOKEN

def generate_prompt_test_gsm8k(data_point):
    return f"""
            Instruction: Give a very short numeric solution with in 30 words or less.
            
            {few_shot_string}
            
            Question: {data_point['question']}.
            Solution:
            """.strip()

import re
ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
INVALID_ANS = "[invalid]"
def extract_the_answer(data_point):
#     print(data_point)
    match = ANS_RE.search(data_point['answer'])
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    else:
        return INVALID_ANS

# X_eval = pd.DataFrame(evaldf.apply(generate_prompt_test_gsm8k, axis=1), columns=["question"])

X_train = pd.DataFrame(traindf.apply(generate_prompt_gsm8k, axis=1), columns=["question"])
X_eval = pd.DataFrame(evaldf.apply(generate_prompt_gsm8k, axis=1), columns=["question"])

y_true = pd.DataFrame(testdf.apply(extract_the_answer, axis=1), columns=["answer"])
X_test = pd.DataFrame(testdf.apply(generate_prompt_test_gsm8k, axis=1), columns=["question"])

train_data = Dataset.from_pandas(X_train)
eval_data = Dataset.from_pandas(X_eval)

In [13]:
eval_data[0]

{'question': "In Professor Plum's biology class there are 40 students. Of those students, 80 percent have puppies. Of those who have puppies, 25% also have parrots. How many students have both puppies and parrots? [SEP] We start with the initial numbers of students, 40 and multiply that by .8 for 40 * 0.8 = <<40*0.8=32>>32 who own puppies.\nThat the number of students with puppies, 32, and multiply that by .25 to find out how many own both puppies and parrots, 32 * 0.25 = <<32*0.25=8>>8 who own puppies and parrots.\nThe answer is <<8=8>>8.\n#### 8<eos>",
 '__index_level_0__': 1297}

In [14]:
X_train = pd.DataFrame(traindf.apply(generate_prompt_gsm8k, axis=1), columns=["question",])

In [15]:
def evaluate(y_true, y_pred, few_shot_number):
    labels = [True, False, None]
    mapping = {True: 1, False: 0, None: 2}
    def map_func(x):
        return x
    
    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)
    
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy of few shot on {few_shot_number} samples is: {accuracy:.3f}')

In [16]:
SUBSTITUTIONS = [
    ('an ', ''), ('a ', ''), ('.$', '$'), ('\\$', ''), (r'\ ', ''), ('\%', '%'),
    (' ', ''), ('mbox', 'text'), (',\\text{and}', ','),
    ('\\text{and}', ','), ('\\text{m}', '\\text{}')
]
REMOVED_EXPRESSIONS = [
    'square', 'ways', 'integers', 'dollars', 'mph', 'inches', 'ft',
    'hours', 'km', 'units', '\\ldots', 'sue', 'points', 'feet',
    'minutes', 'digits', 'cents', 'degrees', 'cm', 'gm', 'pounds',
    'meters', 'meals', 'edges', 'students', 'childrentickets', 'multiples',
    '\\text{s}', '\\text{.}', '\\text{\ns}', '\\text{}^2',
    '\\text{}^3', '\\text{\n}', '\\text{}', r'\mathrm{th}',
    r'^\circ', r'^{\circ}', r'\;', r',\!', '{,}', '"', '\\dots'
]

def normalize_final_answer(final_answer: str) -> str:
    """Normalize a final answer to a quantitative reasoning question."""
    final_answer = final_answer.split('=')[-1]

    for before, after in SUBSTITUTIONS:
        final_answer = final_answer.replace(before, after)
    for expr in REMOVED_EXPRESSIONS:
        final_answer = final_answer.replace(expr, '')

    final_answer = re.sub(r'(.*?)(\$)(.*?)(\$)(.*)', '$\\3$', final_answer)
    final_answer = re.sub(r'(\\text\{)(.*?)(\})', '\\2', final_answer)
    final_answer = re.sub(r'(\\textbf\{)(.*?)(\})', '\\2', final_answer)
    final_answer = re.sub(r'(\\overline\{)(.*?)(\})', '\\2', final_answer)
    final_answer = re.sub(r'(\\boxed\{)(.*)(\})', '\\2', final_answer)

    final_answer = re.sub(
        r'(frac)([^{])(.)', 'frac{\\2}{\\3}', final_answer)
    final_answer = re.sub(
        r'(sqrt)([^{])', 'sqrt{\\2}', final_answer)
    final_answer = final_answer.replace('$', '')

    final_answer = final_answer.replace(',', '')

    return final_answer

In [17]:
y_true_predict = y_true["answer"].tolist()
def predict(X_test, model, tokenizer, y_true):
    y_pred = []
    for i in tqdm(range(len(X_test))):
        prompt = X_test.iloc[i]["question"]
        input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**input_ids, max_new_tokens=100, temperature=0.0)
        result = tokenizer.decode(outputs[0])
        answer = result.split("Solution:")[-1]
        answer = normalize_final_answer(answer)
        pattern = re.compile(fr"[^0-9]*{y_true_predict[i]}[^0-9]+")
        match = re.search(pattern, answer)
#         print(pattern)
#         print(answer)
        if match:
            y_pred.append(y_true_predict[i])
#             print("matched!")
        else:
            y_pred.append(int(y_true_predict[i])+1)
    return y_pred

In [18]:
# y_pred = predict(X_test, model, tokenizer, y_true)

In [19]:
# print("fine-tuned on 10 row ")
# evaluate(y_true, y_pred)

In [20]:
l = [4, 5, 6, 7]
for x in l:
    few_shot_number = x
    few_shot_string = few_shot_string_generator(traindf, x)
    def generate_prompt_test_gsm8k_new(data_point):
        string =f"""
                Instruction: Give a very short numeric solution with in 30 words or less.

                {few_shot_string}

                Question: {data_point['question']}.
                Solution:
                """.strip()
        return string
#     print(few_shot_string)
    y_true = pd.DataFrame(testdf.apply(extract_the_answer, axis=1), columns=["answer"])
    X_test = pd.DataFrame(testdf.apply(generate_prompt_test_gsm8k, axis=1), columns=["question"])
    y_pred = predict(X_test, model, tokenizer, y_true)
    evaluate(y_true, y_pred, few_shot_number)

100%|██████████| 1319/1319 [2:21:23<00:00,  6.43s/it]


Accuracy of few shot on 4 samples is: 0.271


100%|██████████| 1319/1319 [2:19:10<00:00,  6.33s/it]


Accuracy of few shot on 5 samples is: 0.288


100%|██████████| 1319/1319 [2:40:34<00:00,  7.30s/it]


Accuracy of few shot on 6 samples is: 0.276


100%|██████████| 1319/1319 [4:15:33<00:00, 11.62s/it]

Accuracy of few shot on 7 samples is: 0.270


In [21]:
# train_data

In [22]:
# peft_config = LoraConfig(
#     lora_alpha=16,
#     lora_dropout=0,
#     r=64,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
#                     "gate_proj", "up_proj", "down_proj",],
# )

# training_arguments = TrainingArguments(
#     output_dir="logs",
#     num_train_epochs=5,
#     gradient_checkpointing=True,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     optim="paged_adamw_32bit",
#     save_steps=0,
#     logging_steps=1,
#     learning_rate=2e-4,
#     weight_decay=0.001,
#     fp16=True,
#     bf16=False,
#     max_grad_norm=0.3,
#     max_steps=-1,
#     warmup_ratio=0.03,
#     group_by_length=False,
#     evaluation_strategy='steps',
#     eval_steps = 9,
#     eval_accumulation_steps=1,
#     lr_scheduler_type="cosine",
#     report_to="tensorboard",
# )

# trainer = SFTTrainer(
#     model=model,
#     train_dataset=train_data,
#     eval_dataset=eval_data,
#     peft_config=peft_config,
#     dataset_text_field="question",
#     tokenizer=tokenizer,
#     max_seq_length=max_seq_length,
#     args=training_arguments,
#     packing=False,
# )

In [23]:
# # Train model
# trainer.train()

# # Save trained model
# trainer.model.save_pretrained("trained-model-9")

Afterwards, loading the TensorBoard extension and start TensorBoard, pointing to the logs/runs directory, which is assumed to contain the training logs and checkpoints for your model, will allow you to understand how the models fits during the training.

In [24]:
# %load_ext tensorboard
# %tensorboard --logdir logs/runs

The following code will first predict the sentiment labels for the test set using the predict() function. Then, it will evaluate the model's performance on the test set using the evaluate() function. The result now should be impressive with an overall accuracy of over 0.8 and high accuracy, precision and recall for the single sentiment labels. The prediction of the neutral label can still be improved, yet it is impressive how much could be done with little data and some fine-tuning.

In [25]:
# y_pred = predict(X_test, model, tokenizer)
# evaluate(y_true, y_pred)

In [26]:
# evaluation = pd.DataFrame({'question': X_test["question"], 
#                            'y_true':y_true, 
#                            'y_pred': y_pred},
#                          )
# evaluation.to_csv("test_predictions.csv", index=False)